# Notebook 4 — Committor Regression

Train a **LightGBM** regressor to predict the committor probability  
from structural features (O/N pairwise distances + backbone torsions).

**Why LightGBM?** It handles high-dimensional tabular data efficiently,  
requires no feature scaling beyond normalization, and produces interpretable  
SHAP values. Compared to linear regression, XGBoost, and RandomForest, it  
consistently achieved higher R² in our experiments.

**Downsampling:** The committor distribution is strongly bimodal (most frames  
have q≈0 or q≈1). Stratified downsampling (`src/downsample.py`) equalizes the  
class distribution across committor bins so the model learns the transition-state  
region accurately.

**Outputs:**
- `results/feature_importance.jpg`
- `results/shap_summary.jpg`
- `results/true_vs_pred_train.jpg` / `results/true_vs_pred_test.jpg`

In [ ]:
import yaml
import os
import sys
sys.path.insert(0, "..")
import pandas as pd

from src.regression import regression
from src.plot_importances import plot_importances, plot_shap, plot_true_vs_pred

os.makedirs("../results", exist_ok=True)

In [ ]:
# --- Select system ---
with open("../config/chignolin.yaml") as f:   # change to ww_domain.yaml for WW domain
    cfg = yaml.safe_load(f)

SYSTEM = cfg["system"]
N = 40   # cluster count used when computing committors

CLUSTER_PROB_CSV = f"../data/{SYSTEM}/cluster_{N}_prob.csv"
DIST_CSV         = f"../data/{SYSTEM}/distances_ON.csv"
TORS_CSV         = f"../data/{SYSTEM}/backbone_torsions.csv"

print(f"Committor CSV: {CLUSTER_PROB_CSV}")
print(f"Features:      {DIST_CSV}, {TORS_CSV}")

In [ ]:
# Load pre-computed feature matrices (from Notebook 3)
df_dist = pd.read_csv(DIST_CSV)
df_tors = pd.read_csv(TORS_CSV)
df_features = pd.concat([df_dist, df_tors], axis=1)
print(f"Feature matrix: {df_features.shape}")

## Train the regressor

In [ ]:
# down=1 uses the full dataset; set to e.g. 0.1 for 10% stratified downsample
score, model, X_train, X_test, y_train, y_test = regression(
    df_features,
    committor_csv=CLUSTER_PROB_CSV,
    down=1
)
print(f"\nTest R²: {score:.4f}")

## Feature importance and SHAP

In [ ]:
plot_importances(model, filename="../results/feature_importance.jpg")

In [ ]:
plot_shap(model, X_test, filename="../results/shap_summary.jpg")

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

plot_true_vs_pred(y_train, y_pred_train,
                  filename="../results/true_vs_pred_train.jpg",
                  title="Train — True vs Predicted Committor")
plot_true_vs_pred(y_test, y_pred_test,
                  filename="../results/true_vs_pred_test.jpg",
                  title="Test — True vs Predicted Committor")

## Effect of downsampling on R²

Sweep the downsampling fraction to see how much of the data is needed.

In [ ]:
import matplotlib.pyplot as plt

fractions = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
scores = []
for frac in fractions:
    s, *_ = regression(df_features, committor_csv=CLUSTER_PROB_CSV, down=frac)
    scores.append(s)
    print(f"  down={frac:.2f}  R²={s:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(fractions, scores, 'o-')
plt.xlabel('Fraction of data used')
plt.ylabel('Test R²')
plt.title('R² vs. downsampling fraction')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()